# Voltametria de Corrente Amostrada

# Bibliotecas Necessárias:

In [ ]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from scipy import special
from ipywidgets import Text, Button, HBox, VBox, Output
from IPython.display import display, clear_output

# Definindo o Modelo:

In [ ]:
model = pybamm.BaseModel()

co = pybamm.Variable("Concentração de O", domain="electrolyte")
cr = pybamm.Variable("Concentração de R", domain="electrolyte")

eta = pybamm.Parameter("Sobrepotencial Aplicado [V]")
F   = pybamm.Parameter("Constante de Faraday [C.mol-1]")
R   = pybamm.Parameter("Constante dos Gases [J.K-1.mol-1]")
T   = pybamm.Parameter("Temperatura [K]")

No = -pybamm.grad(co)   # fluxo difusional de O
Nr = -pybamm.grad(cr)   # fluxo difusional de R

model.rhs = {
    co: -pybamm.div(No),  # lei de Fick para O (eq. 1.0a)
    cr: -pybamm.div(Nr),  # lei de Fick para R (eq. 1.0b)
}

# condições iniciais
model.initial_conditions = {
    co: pybamm.Scalar(1),  # c_O(x,0) = 1: O uniforme e adimensionalizado
    cr: pybamm.Scalar(0),  # c_R(x,0) = 0: R inicialmente ausente
}

# condições de contorno — Dirichlet
# esquerda (x=0): interface eletrodo/solução — equilíbrio de Nernst
# direita  (x=6): seio da solução            — difusão semi-infinita
f    = F / (R * T)
teta = pybamm.exp(f * eta)

model.boundary_conditions = {
    co: {
        "left":  (teta / (1 + teta), "Dirichlet"),  # eq. 1.3a: c_O(0,t) = θ/(1+θ)
        "right": (pybamm.Scalar(1),  "Dirichlet"),  # eq. 1.2a: c_O(∞,t) = 1
    },
    cr: {
        "left":  (1 / (1 + teta),   "Dirichlet"),  # eq. 1.3b: c_R(0,t) = 1/(1+θ)
        "right": (pybamm.Scalar(0),  "Dirichlet"),  # eq. 1.2b: c_R(∞,t) = 0
    },
}

model.variables = {
    "Concentração de O": co,
    "Concentração de R": cr,
    "Fluxo de O": No,
    "Fluxo de R": Nr,
}

param = pybamm.ParameterValues(
    {
        "Sobrepotencial Aplicado [V]": "[input]",
        "Constante de Faraday [C.mol-1]": 96485.3,
        "Constante dos Gases [J.K-1.mol-1]": 8.31446,
        "Temperatura [K]": 298.15,
    }
)

# ── geometria e malha ────────────────────────────────────────────────────────
x_var = pybamm.SpatialVariable("x", domain=["electrolyte"], coord_sys="cartesian")

geometry = {"electrolyte": {x_var: {"min": pybamm.Scalar(0), "max": pybamm.Scalar(6)}}}

submesh_types = {"electrolyte": pybamm.Uniform1DSubMesh}
var_pts       = {x_var: 400}
mesh          = pybamm.Mesh(geometry, submesh_types, var_pts)

spatial_methods = {"electrolyte": pybamm.FiniteVolume()}
disc = pybamm.Discretisation(mesh, spatial_methods)

param.process_model(model)
param.process_geometry(geometry)
disc.process_model(model)

# Gráfico Interativo:

In [ ]:
F_NUM = 38.9217  # f = F/(RT) a 298.15 K [V⁻¹]

solver = pybamm.ScipySolver()

# Pontos de tempo: a corrente é amostrada apenas em t=1, mas o solver precisa
# de uma malha temporal para convergir corretamente até esse instante.
# Reduzir abaixo de 200 pode comprometer a precisão — ajuste com cautela.
t = np.linspace(1e-5, 1, 200)

saida = Output()

def plotar(ei_str, ef_str, e0_str, npts_str):
    with saida:
        clear_output(wait=True)

        try:
            ei   = float(ei_str)
            ef   = float(ef_str)
            e0   = float(e0_str)
            npts = int(npts_str)
        except ValueError:
            print("Por favor, insira valores numéricos válidos.")
            return

        if ei >= ef:
            print("O potencial inicial deve ser menor que o potencial final.")
            return

        if npts < 2:
            print("O número de pontos deve ser maior que 1.")
            return

        # potenciais dos saltos — linspace evita imprecisões do arange com floats
        es    = np.linspace(ei, ef, npts)
        i_smp = []

        for e_ap in es:
            solucao = solver.solve(
                model, t,
                inputs={"Sobrepotencial Aplicado [V]": e_ap - e0}
            )
            No_sol = solucao["Fluxo de O"]
            # corrente amostrada em t=1, normalizada pela corrente limite de Cottrell
            i_smp.append(-No_sol(1, x=0) * np.sqrt(np.pi))

        fig, ax = plt.subplots(figsize=(6.5, 4))

        ax.plot(es, i_smp, "r-", linewidth=1.5)
        ax.set_xlabel("E / V")
        ax.set_ylabel("Corrente")
        ax.set_xlim([ei, ef])  # eixo X: negativo à esquerda, positivo à direita
        ax.set_title("Voltametria de Corrente Amostrada")

        plt.tight_layout()
        display(fig)
        plt.close(fig)

# ── widgets ──────────────────────────────────────────────────────────────────
campo_ei = Text(
    value="-0.5",
    description="Potencial Inicial [V]:",
    style={"description_width": "initial"},
)

campo_ef = Text(
    value="0.5",
    description="Potencial Final [V]:",
    style={"description_width": "initial"},
)

campo_e0 = Text(
    value="0",
    description="Potencial Padrão [V]:",
    style={"description_width": "initial"},
)

campo_npts = Text(
    value="5",
    description="Número de Pontos:",
    style={"description_width": "initial"},
)

botao = Button(description="Recalcular", button_style="success")

def ao_clicar(b):
    plotar(campo_ei.value, campo_ef.value, campo_e0.value, campo_npts.value)

botao.on_click(ao_clicar)

interface = VBox([
    HBox([campo_ei, campo_ef]),
    HBox([campo_e0, campo_npts, botao]),
    saida,
])
display(interface)

# gráfico inicial
plotar(campo_ei.value, campo_ef.value, campo_e0.value, campo_npts.value)